In [23]:
import pandas as pd
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from src.data_loader import load_data

df = load_data("../data/raw/MachineLearningRating_v3.txt")

c:\Users\Alienware\Desktop\kaim works\insurance-risk-analytics\src\data_loader.py:4: DtypeWarning: Columns (32,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, sep="|")


In [11]:
df["TotalClaims"] = pd.to_numeric(df["TotalClaims"], errors="coerce")
df["TotalPremium"] = pd.to_numeric(df["TotalPremium"], errors="coerce")

In [12]:
df["HasClaim"] = (df["TotalClaims"] > 0).astype(int)

In [13]:
df_reg = df[df["HasClaim"] == 1].copy()

In [14]:
df["VehicleAge"] = 2026 - df["RegistrationYear"]
df["Margin"] = df["TotalPremium"] - df["TotalClaims"]

In [19]:
df = pd.get_dummies(df, drop_first=True)

In [20]:
from sklearn.model_selection import train_test_split

# Use numeric features only to avoid extremely high-dimensional one-hot encoding
X = df.select_dtypes(include=["number"]).drop(["TotalClaims", "HasClaim"], axis=1)

y = df["HasClaim"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [32]:
from sklearn.metrics import mean_squared_error, r2_score
from src.modeling import get_regression_models
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
import numpy as np

models = get_regression_models()
# Wrap each model with an imputer to handle NaNs
models = {name: make_pipeline(SimpleImputer(strategy='median'), model) for name, model in models.items()}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    results.append([name, rmse, r2])

c:\Users\Alienware\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\impute\_base.py:635: UserWarning: Skipping features without any observed values: ['NumberOfVehiclesInFleet']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\Alienware\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\impute\_base.py:635: UserWarning: Skipping features without any observed values: ['NumberOfVehiclesInFleet']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\Alienware\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\impute\_base.py:635: UserWarning: Skipping features without any observed values: ['NumberOfVehiclesInFleet']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\Alienware\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\impute\_base.py:635: UserWarning: Skipp

In [27]:
pd.DataFrame(results, columns=["Model", "RMSE", "R2"])

,Model,RMSE,R2
0,LinearRegression,0.045260,0.292713
1,RandomForest,0.001413,0.999311
2,XGBoost,0.000919,0.999708


In [28]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [29]:
clf_models = get_classification_models()

clf_results = []

for name, model in clf_models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    clf_results.append([
        name,
        accuracy_score(y_test, preds),
        precision_score(y_test, preds),
        recall_score(y_test, preds),
        f1_score(y_test, preds)
    ])

NameError: name 'get_classification_models' is not defined

In [ ]:
pd.DataFrame(clf_results,
             columns=["Model","Accuracy","Precision","Recall","F1"])

In [31]:
df["Predicted_Severity"] = best_reg_model.predict(X)
df["Claim_Probability"] = best_clf_model.predict_proba(X)[:,1]

df["Predicted_Premium"] = (
    df["Claim_Probability"] * df["Predicted_Severity"]
    + 50   # expense loading
    + 100  # profit margin
)

NameError: name 'best_reg_model' is not defined

In [30]:
import shap

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)

ModuleNotFoundError: No module named 'shap'